In [ ]:
%matplotlib qt

Point 1: x=96.8, y=53.5
Point 2: x=94.7, y=98.4
Point 3: x=98.9, y=136.3
Point 4: x=104.5, y=162.2
Point 5: x=123.5, y=194.5
Point 6: x=140.3, y=215.5
Point 1: x=110.8, y=54.9
Point 2: x=101.7, y=97.7
Point 3: x=106.6, y=136.3
Point 4: x=108.7, y=160.1
Point 5: x=126.3, y=191.7
Point 1: x=115.7, y=53.5
Point 2: x=106.6, y=102.6
Point 3: x=108.0, y=134.9
Point 4: x=122.8, y=169.2
Point 1: x=98.2, y=57.0
Point 2: x=95.4, y=73.8
Point 3: x=93.3, y=88.6
Point 4: x=91.2, y=111.0
Point 5: x=92.6, y=130.6
Point 6: x=95.4, y=140.5
Point 7: x=103.1, y=151.7
Point 8: x=162.0, y=248.5


In [ ]:
import sunpy.map
import matplotlib.pyplot as plt
import glob
import pandas as pd
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord
from scipy.signal import find_peaks

#1. find the closest -short file to a given regular file

def find_closest_short(regular_map, short_dir='../data/20260820T154500'):
    short_files = sorted(glob.glob(f'{short_dir}/solo_L2_eui-hrieuv174-image-short_*.fits'))
    short_maps = [sunpy.map.Map(f) for f in short_files]
    short_times = pd.to_datetime([sm.date.iso for sm in short_maps])

    target = pd.Timestamp(regular_map.date.iso)
    closest_idx = np.abs(short_times - target).argmin()
    return short_maps[closest_idx]

#2. make the 150 x 150 arcsec zoom

def make_zoom(map_obj):

    zoom_width = 150 * u.arcsec
    zoom_height = 150 * u.arcsec

    bottom_left = SkyCoord(
        -1370 * u.arcsec - zoom_width / 2,
        375 * u.arcsec - zoom_height / 2,
        frame=map_obj.coordinate_frame
    )

    top_right = SkyCoord(
        -1370 * u.arcsec + zoom_width / 2,
        375 * u.arcsec + zoom_height / 2,
        frame=map_obj.coordinate_frame
    )

    return map_obj.submap(
        bottom_left,
        top_right=top_right
    )

#3. pick slice points on the map

def pick_points(map_obj, n_points=None):

    zoomed_map = make_zoom(map_obj)

    fig = plt.figure(figsize=(7, 7))
    ax = fig.add_subplot(projection=zoomed_map.wcs)

    zoomed_map.plot(axes=ax)

    ax.set_title(
        f"Click points for slice ({map_obj.date.iso[11:19]})"
    )

    clicked = []

    def onclick(event):

        if event.inaxes != ax:
            return

        clicked.append(
            (event.xdata, event.ydata)
        )

        ax.plot(
            event.xdata,
            event.ydata,
            "ro",
            markersize=5
        )

        if len(clicked) > 1:

            x = [p[0] for p in clicked]
            y = [p[1] for p in clicked]

            ax.plot(
                x,
                y,
                "r-",
                linewidth=2
            )

        fig.canvas.draw_idle()

        print(
            f"Point {len(clicked)}: "
            f"x={event.xdata:.1f}, "
            f"y={event.ydata:.1f}"
        )

    fig.canvas.mpl_connect(
        "button_press_event",
        onclick
    )

    return zoomed_map, clicked
# 4. extract intensity profile along the path

def extract_profile(map_obj, points):
    x_pix, y_pix = [], []
    for i in range(len(points) - 1):
        p1 = map_obj.world_to_pixel(points[i])
        p2 = map_obj.world_to_pixel(points[i+1])
        x1, y1 = p1[0].value, p1[1].value
        x2, y2 = p2[0].value, p2[1].value
        n = int(np.hypot(x2 - x1, y2 - y1))
        xs = np.linspace(x1, x2, n)
        ys = np.linspace(y1, y2, n)
        if i > 0:
            xs, ys = xs[1:], ys[1:]  # drop duplicate joint point
        x_pix.append(xs)
        y_pix.append(ys)

    x_pixels = np.round(np.concatenate(x_pix)).astype(int)
    y_pixels = np.round(np.concatenate(y_pix)).astype(int)

    valid = ((x_pixels >= 0) & (x_pixels < map_obj.data.shape[1]) &
             (y_pixels >= 0) & (y_pixels < map_obj.data.shape[0]))
    x_pixels, y_pixels = x_pixels[valid], y_pixels[valid]

    intensity = map_obj.data[y_pixels, x_pixels]
    return x_pixels, y_pixels, intensity



In [ ]:
# 5. plot slice line, intensity profile, and peaks, for one map

def analyze_map(map_obj, points, label):

    map_obj = make_zoom(map_obj)

    x_pixels, y_pixels, intensity = extract_profile(
        map_obj,
        points
    )

    distance = np.arange(len(intensity))

    peaks, _ = find_peaks(
        intensity,
        prominence=np.std(intensity) * 0.005,
        distance=2
    )


    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(projection=map_obj.wcs)

    map_obj.plot(axes=ax)

    coords = SkyCoord(
        [p.Tx for p in points],
        [p.Ty for p in points],
        frame=map_obj.coordinate_frame
    )

    ax.plot_coord(
        coords,
        color="red",
        linewidth=2,
        marker="o",
        markersize=4
    )

    ax.set_title(
        f"Slice location ({label}) — {map_obj.date.iso[11:19]}"
    )

    plt.tight_layout()

    plt.savefig(
        f"../figures/eui_slice_location_{label}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

  

    plt.figure(figsize=(8, 4))

    plt.plot(
        distance,
        intensity,
        drawstyle="steps-mid",
        label="Intensity profile"
    )

    plt.plot(
        distance[peaks],
        intensity[peaks],
        "ro",
        markersize=5,
        label="Detected peaks"
    )

    plt.xlabel("Distance along slice (pixels)")
    plt.ylabel("Intensity [DN/s]")
    plt.title(f"EUI intensity profile ({label})")

    plt.legend()
    plt.tight_layout()

    plt.savefig(
        f"../figures/eui_intensity_profile_{label}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()



    peak_coords = map_obj.pixel_to_world(
        x_pixels[peaks] * u.pix,
        y_pixels[peaks] * u.pix
    )

    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(projection=map_obj.wcs)

    map_obj.plot(axes=ax)

    ax.plot_coord(
        peak_coords,
        marker="o",
        linestyle="none",
        markersize=4,
        color="red"
    )

    ax.set_title(
        f"Peaks on image ({label}) — {map_obj.date.iso[11:19]}"
    )

    plt.tight_layout()

    plt.savefig(
        f"../figures/eui_slice_peaks_{label}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

    return intensity, peaks

In [ ]:


regular_map = sunpy.map.Map(
    "../data/20260820T154500/"
    "solo_L2_eui-hrieuv174-image_20260820T162956285_V00.fits"
)

short_map = find_closest_short(regular_map)

zoomed_regular, clicked = pick_points(regular_map)

print(f"Picked {len(clicked)} points")

Picked 0 points


In [22]:
print(f"Number of points: {len(clicked)}")
print(clicked)

Number of points: 8
[(np.float64(98.20779220779221), np.float64(57.00649350649353)), (np.float64(95.4025974025974), np.float64(73.83766233766237)), (np.float64(93.2987012987013), np.float64(88.56493506493509)), (np.float64(91.19480519480518), np.float64(111.00649350649353)), (np.float64(92.5974025974026), np.float64(130.64285714285717)), (np.float64(95.4025974025974), np.float64(140.46103896103898)), (np.float64(103.11688311688312), np.float64(151.6818181818182)), (np.float64(162.02597402597402), np.float64(248.46103896103898))]


In [ ]:


points = [
    zoomed_regular.pixel_to_world(
        x * u.pix,
        y * u.pix
    )
    for x, y in clicked
]

for i, point in enumerate(points):

    print(
        f"Point {i+1}: "
        f"Tx = {point.Tx.to(u.arcsec):.2f}, "
        f"Ty = {point.Ty.to(u.arcsec):.2f}"
    )

Point 1: Tx = -1395.13 arcsec, Ty = 300.71 arcsec
Point 2: Tx = -1398.71 arcsec, Ty = 308.30 arcsec
Point 3: Tx = -1401.67 arcsec, Ty = 315.00 arcsec
Point 4: Tx = -1405.67 arcsec, Ty = 325.34 arcsec
Point 5: Tx = -1407.63 arcsec, Ty = 334.83 arcsec
Point 6: Tx = -1407.61 arcsec, Ty = 339.85 arcsec
Point 7: Tx = -1405.46 arcsec, Ty = 346.20 arcsec
Point 8: Tx = -1390.50 arcsec, Ty = 399.89 arcsec


In [ ]:


intensity_reg, peaks_reg = analyze_map(
    regular_map,
    points,
    label="regular"
)

intensity_short, peaks_short = analyze_map(
    short_map,
    points,
    label="short"
)